In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import QuantileTransformer
import joblib

df = pd.read_csv('dataset/Teen_Mental_Health_Dataset.csv')
nivel_map = {'low': 0, 'medium': 1, 'high': 2}

df.shape

(1200, 13)

In [ ]:
# Ordinal Maping
df['social_interaction_level'] = df['social_interaction_level'].map(nivel_map)

df['total_digital_load'] = df['daily_social_media_hours'] + df['screen_time_before_sleep']

df['sleep_quality_ratio'] = df['sleep_hours'] / (df['screen_time_before_sleep'] + 1)

df['mental_health_risk'] = df['stress_level'] + df['anxiety_level'] + df['addiction_level']

df['lifestyle_balance'] = (df['physical_activity'] + df['social_interaction_level']) / (df['daily_social_media_hours'] + 1)

print("Nuevas dimensiones del dataset:", df.shape)
df.head(3)

Nuevas dimensiones del dataset: (1200, 17)


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,total_digital_load,sleep_quality_ratio,mental_health_risk,lifestyle_balance
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,0,2,2,1,0,10.8,1.897436,5,0.168539
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,2,8,1,10,0,4.8,2.051282,19,0.965517
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,2,2,4,2,0,1.8,5.066667,8,0.869565


La columna de academic_performace va a ser la nueva y (output varaible), en el dataset de manera original se usa depression_label como el output. Sin embargo esta columna tiene un desbalance, más del 97% de las instacias cuentan con 0. Por ese motivo hago un drop de la columna y del target.

In [34]:
X = df.drop(columns=['academic_performance', 'depression_label'])
y = df['academic_performance']

distribucion_absoluta = df['depression_label'].value_counts()
distribucion_porcentual = df['depression_label'].value_counts(normalize=True) * 100

pd.DataFrame({
    'Frecuencia Absoluta': distribucion_absoluta,
    'Proporción Porcentual (%)': distribucion_porcentual.round(2)
})

,Frecuencia Absoluta,Proporción Porcentual (%)
depression_label,,
0,1169,97.42
1,31,2.58


Para limpiar los datos primero realizo el split de los datos de train(80%) y test(20%), hacer la separación desde este punto tiene sentido ya que en posteriormente vamos a usar una técnica de escalamiento de datos, si calculamos medias numéricas o rangos utilizando el archivo completo antes del split, el set de entrenamiento tendría parámetros estadísticos del set de prueba.

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño del set de train: {X_train.shape[0]} instancias")
print(f"Tamaño del set de test: {X_test.shape[0]} instancias")

Tamaño del set de train: 960 instancias
Tamaño del set de test: 240 instancias


Mostramos los tipos de datos del dataset para hacer una limpieza/homologar los tipos de datos.

In [36]:
X_train.dtypes

age                           int64
gender                          str
daily_social_media_hours    float64
platform_usage                  str
sleep_hours                 float64
screen_time_before_sleep    float64
physical_activity           float64
social_interaction_level      int64
stress_level                  int64
anxiety_level                 int64
addiction_level               int64
total_digital_load          float64
sleep_quality_ratio         float64
mental_health_risk            int64
lifestyle_balance           float64
dtype: object

La columna de nivel de interaccion social tiene string que son ordinales, por lo que vamos a hacer un ordinal mapping esta columna, para las columnas de genero y plataforma vamos a usar un one-hot enconding. AL final forzamos que los booleanos sean 0/1 y no True/False que es lo que usa Python por default.

In [37]:
# One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train, columns=['gender', 'platform_usage'], drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=['gender', 'platform_usage'], drop_first=True)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Convertimos los booleanos a enteros
for col in X_train_encoded.columns:
    if X_train_encoded[col].dtype == bool or X_train_encoded[col].dtype == 'bool':
        X_train_encoded[col] = X_train_encoded[col].astype(int)
        X_test_encoded[col] = X_test_encoded[col].astype(int)

X_train_encoded.dtypes

age                           int64
daily_social_media_hours    float64
sleep_hours                 float64
screen_time_before_sleep    float64
physical_activity           float64
social_interaction_level      int64
stress_level                  int64
anxiety_level                 int64
addiction_level               int64
total_digital_load          float64
sleep_quality_ratio         float64
mental_health_risk            int64
lifestyle_balance           float64
gender_male                   int64
platform_usage_Instagram      int64
platform_usage_TikTok         int64
dtype: object

Podemos verificar que ya los tipos de objetos son numericos int/float, lo que vamos a realizar ahora es escalar los features numericos del resto de columnas.

In [38]:
columnas_numericas = ['age', 'social_interaction_level', 'stress_level',
                     'anxiety_level', 'addiction_level']

columns_to_transform = ['daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'physical_activity', 'total_digital_load', 'sleep_quality_ratio', 'mental_health_risk',
                        'lifestyle_balance']

scaler = StandardScaler()
quantile_scaler = QuantileTransformer(n_quantiles=100, output_distribution='normal')
X_train_ready = X_train_encoded.copy()
X_test_ready = X_test_encoded.copy()

X_train_ready[columnas_numericas] = scaler.fit_transform(X_train_encoded[columnas_numericas])
X_train_ready[columns_to_transform] = quantile_scaler.fit_transform(X_train_encoded[columns_to_transform])

X_test_ready[columnas_numericas] = scaler.transform(X_test_encoded[columnas_numericas])
X_test_ready[columns_to_transform] = quantile_scaler.transform(X_test_encoded[columns_to_transform])


Podemos observas que ahora todos los datos son de tipo float, menos aquellos que son datos binarios (a los que les relaizamos el one-hot encoding)

In [39]:
X_test_ready.dtypes

age                         float64
daily_social_media_hours    float64
sleep_hours                 float64
screen_time_before_sleep    float64
physical_activity           float64
social_interaction_level    float64
stress_level                float64
anxiety_level               float64
addiction_level             float64
total_digital_load          float64
sleep_quality_ratio         float64
mental_health_risk          float64
lifestyle_balance           float64
gender_male                   int64
platform_usage_Instagram      int64
platform_usage_TikTok         int64
dtype: object

Por ultimo exportamos los datos en csv tanto para train como para test.

In [40]:
train_final_export = X_train_ready.copy()
train_final_export['target_academic_performance'] = y_train

test_final_export = X_test_ready.copy()
test_final_export['target_academic_performance'] = y_test

train_final_export.to_csv('dataset/teen_train_clean.csv', index=False)
test_final_export.to_csv('dataset/teen_test_clean.csv', index=False)

train_final_export.head(5) 

,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,total_digital_load,sleep_quality_ratio,mental_health_risk,lifestyle_balance,gender_male,platform_usage_Instagram,platform_usage_TikTok,target_academic_performance
331,0.020422,0.595113,0.967422,-0.362241,0.834427,-1.195656,1.564665,-1.598783,-0.553219,0.444660,0.911804,-0.335333,-0.568987,0,1,0,3.27
409,-0.469701,0.430727,0.589456,0.731217,0.389414,0.043901,1.564665,1.560029,-1.256185,0.698526,-0.222135,1.073988,-0.045799,0,1,0,2.84
76,-1.449946,-2.322575,0.335333,0.050661,-0.282216,1.283458,-0.141854,-0.194867,1.555678,-1.639976,0.195841,0.650837,2.104585,1,0,1,2.55
868,0.020422,0.282216,-1.508944,-0.242903,-1.008673,1.283458,0.540753,0.507092,1.555678,0.165327,-0.835846,1.508944,0.062132,0,0,1,2.54
138,-0.959824,0.698526,-0.242903,-0.698526,-0.165327,0.043901,0.882057,-1.247804,-0.904702,0.444660,0.531471,-0.698526,-0.409231,0,1,0,3.81


In [41]:
joblib.dump(scaler, 'scaler_gpa.pkl')
joblib.dump(quantile_scaler, 'quantile_scaler_gpa.pkl')

['quantile_scaler_gpa.pkl']